# KLTN — ABSA Token Merging · chạy full luồng trên Kaggle

Notebook này clone code từ GitHub rồi chạy `run_all.py` — toàn bộ pipeline
(train ATE → APC các biến thể → eval → hình → báo cáo).

## Trước khi chạy — bật 2 thứ trong panel bên phải

| Mục | Giá trị |
|---|---|
| **Accelerator** | `GPU T4 x2` hoặc `GPU P100` |
| **Internet** | `On` (bắt buộc — để `git clone` và tải model HuggingFace) |

## Thứ tự chạy

1. Cell **Cấu hình** → sửa `BRANCH` nếu cần
2. Cell **Clone / pull** → lấy code mới nhất
3. Cell **Cài thư viện** → chỉ lâu ở lần đầu
4. Cell **Kiểm tra môi trường** → xác nhận thấy GPU
5. Cell **SMOKE TEST** ← **chạy cái này trước**, ~5–10 phút, xác nhận cả đường ống chạy được
6. Cell **Chạy thật** → chỉ chạy khi smoke test xanh
7. Cell **Xem báo cáo** / **Đóng gói kết quả**

> **Phiên Kaggle tối đa 9–12 giờ.** Lượt đầy đủ (21 biến thể × 3 seed × 2 backbone)
> KHÔNG chạy xong trong một phiên. Hãy dùng `--resume` và chia nhiều phiên, hoặc
> thu hẹp phạm vi ở cell **Chạy thật**.

## 1 · Cấu hình

In [ ]:
import os
from pathlib import Path

# ── Nguồn code ────────────────────────────────────────────────────────────
REPO_URL  = "https://github.com/hotuyen21pt/KLTN-Token-Merging.git"
BRANCH    = "tuyen"
REPO_NAME = "KLTN-Token-Merging"

# Repo private? Thêm Kaggle Secret tên GITHUB_TOKEN (Add-ons → Secrets).
# Repo public thì bỏ qua, cell clone tự chạy không cần token.
USE_TOKEN = False

# ── Thư mục làm việc ──────────────────────────────────────────────────────
# /kaggle/working được giữ lại sau phiên (giới hạn ~20 GB).
# /kaggle/temp bị xoá khi hết phiên → để cache model cho khỏi tốn quota output.
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO_DIR  = WORK_ROOT / REPO_NAME
CACHE_DIR = Path("/kaggle/temp/hf") if Path("/kaggle").exists() else WORK_ROOT / ".hf"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["MPLBACKEND"] = "Agg"   # Kaggle không có màn hình

print(f"Repo dir  : {REPO_DIR}")
print(f"HF cache  : {CACHE_DIR}")
print(f"Branch    : {BRANCH}")

## 2 · Clone / pull code từ GitHub

Chạy được nhiều lần:

- **Lần đầu** → `git clone`
- **Lần sau** → sửa lại `origin`, `git fetch --all --prune`, rồi `git reset --hard origin/<BRANCH>`

`reset --hard` đảm bảo code trong phiên khớp *chính xác* với GitHub — mọi thay đổi
cục bộ trong `/kaggle/working` sẽ bị bỏ. Kết quả thực nghiệm nằm ngoài vùng git
theo dõi (đã `.gitignore`) nên **không** bị xoá; muốn chắc thì chạy cell đóng gói
ở cuối trước khi pull lại.

In [ ]:
import subprocess, sys, shutil

def sh(cmd, cwd=None, check=True):
    """Chạy lệnh, in output theo thời gian thực (lượt full chạy nhiều giờ)."""
    cmd = [str(c) for c in cmd]
    print(f"$ {' '.join(cmd)}", flush=True)
    proc = subprocess.Popen(
        cmd, cwd=cwd, text=True, encoding="utf-8", errors="replace",
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if check and proc.returncode != 0:
        raise RuntimeError(f"Lệnh thất bại (exit {proc.returncode}): {' '.join(cmd)}")
    return proc.returncode


# ── Dựng URL (kèm token nếu repo private) ─────────────────────────────────
clone_url = REPO_URL
if USE_TOKEN:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    clone_url = REPO_URL.replace("https://", f"https://{token}@")
    print("Dùng GITHUB_TOKEN từ Kaggle Secrets")


if (REPO_DIR / ".git").is_dir():
    print(f"Đã có repo tại {REPO_DIR} → cập nhật\n")
    sh(["git", "remote", "set-url", "origin", clone_url], cwd=REPO_DIR)
    sh(["git", "fetch", "--all", "--prune", "--tags"], cwd=REPO_DIR)
    sh(["git", "checkout", "-B", BRANCH, f"origin/{BRANCH}"], cwd=REPO_DIR)
    sh(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR)
    # Dọn file rác do git tạo, GIỮ LẠI kết quả (đã nằm trong .gitignore)
    sh(["git", "clean", "-fd"], cwd=REPO_DIR)
else:
    print(f"Chưa có repo → clone nhánh {BRANCH}\n")
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    sh(["git", "clone", "--branch", BRANCH, "--single-branch", clone_url, str(REPO_DIR)])

# Giấu token khỏi .git/config sau khi xong
if USE_TOKEN:
    sh(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO_DIR)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print()
sh(["git", "log", "--oneline", "-5"], cwd=REPO_DIR)
sh(["git", "status", "--short", "--branch"], cwd=REPO_DIR)
print(f"\nThư mục hiện tại: {os.getcwd()}")

## 3 · Cài thư viện

Image Kaggle đã có `torch`, `transformers`, `scikit-learn`, `matplotlib`, `pandas`.
Cell này bù những gói còn thiếu.

**`Levenshtein` là bắt buộc** — `src/normalization.py` import nó, nên thiếu là
stage `ate`, `ate_infer`, `gas` chết ngay với `ModuleNotFoundError`. Cell cài
từng gói một lệnh riêng (gộp chung thì một gói hỏng làm pip bỏ cả lô) và
**dừng hẳn** nếu gói bắt buộc vẫn thiếu, thay vì chỉ cảnh báo rồi đi tiếp.

`pyabsa` là tuỳ chọn — không có thì các stage chính vẫn chạy.


In [ ]:
FULL_INSTALL = False   # True = cài đúng requirements.txt (lâu, dễ đụng torch của Kaggle)

# (tên pip, tên import, bắt buộc?)
PACKAGES = [
    ("python-Levenshtein>=0.25.0", "Levenshtein", True),
    ("seaborn",                    "seaborn",     True),
    ("tqdm",                       "tqdm",        True),
    ("pyabsa>=2.4.0,<3",           "pyabsa",      False),
]

if FULL_INSTALL:
    sh([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
else:
    for spec, _mod, required in PACKAGES:
        # Mỗi gói một lệnh pip riêng: gộp chung thì một gói resolve hỏng
        # sẽ làm pip bỏ luôn cả lô (đây chính là lỗi làm mất Levenshtein).
        rc = sh([sys.executable, "-m", "pip", "install", "-q", spec], check=False)
        if rc != 0:
            print(f"  [!] pip install {spec} → exit {rc}"
                  f"{'  (BẮT BUỘC)' if required else '  (tuỳ chọn, bỏ qua được)'}")

print("\n" + "=" * 60)
missing_required = []
for spec, mod, required in PACKAGES + [
    ("torch", "torch", True), ("transformers", "transformers", True),
    ("scikit-learn", "sklearn", True), ("matplotlib", "matplotlib", True),
    ("pandas", "pandas", True),
]:
    try:
        m = __import__(mod)
        print(f"  {mod:<14}: {getattr(m, '__version__', 'ok')}")
    except Exception as e:
        tag = "THIẾU (BẮT BUỘC)" if required else "thiếu (tuỳ chọn)"
        print(f"  {mod:<14}: {tag} — {type(e).__name__}")
        if required:
            missing_required.append((spec, mod))
print("=" * 60)

if missing_required:
    raise SystemExit(
        "DỪNG — thiếu gói bắt buộc: "
        + ", ".join(m for _, m in missing_required)
        + "\nCài tay rồi RESTART kernel:\n"
        + "\n".join(f"  !pip install {s}" for s, _ in missing_required)
    )
print("Đủ thư viện bắt buộc.")


## 4 · Kiểm tra môi trường

`run_all.py --list` in ra 12 stage và 21 biến thể; `--stages env` ghi snapshot
môi trường ra `reports/00_environment.txt`.

In [ ]:
sh(["nvidia-smi"], check=False)
print()
sh([sys.executable, "run_all.py", "--list"])

## 5 · SMOKE TEST — chạy cái này trước

`--smoke` dựng một dataset tí hon (72 train / 32 dev / 32 test, phủ đủ 6 category × 3 sentiment) rồi chạy **toàn bộ**
đường ống với 1 epoch, 1 seed, 3 biến thể (phủ đủ 3 nhánh code: post-ToMe resize,
post-ToMe compact, pre-ToMe).

Mọi output đi vào `smoke_run/` — **không đè lên kết quả thật**.

Mất khoảng **5–10 phút** trên GPU (phần lớn là tải `t5-base` + `bert-base-uncased`
lần đầu). Nếu cell này xanh thì lượt đầy đủ cũng sẽ chạy được.

In [ ]:
rc = sh([sys.executable, "run_all.py", "--smoke", "--skip", "uos"], check=False)
print(f"\n{'='*70}")
print("SMOKE TEST: " + ("ĐẠT — đường ống chạy được" if rc == 0 else f"HỎNG (exit {rc})"))
print("=" * 70)
print("Log chi tiết : smoke_run/reports/logs/<stage>.log")
print("Báo cáo      : smoke_run/reports/REPORT.md")

In [ ]:
# Bảng trạng thái từng stage của lượt smoke
import json

status_file = REPO_DIR / "smoke_run" / "reports" / "run_status.json"
if status_file.is_file():
    data = json.loads(status_file.read_text(encoding="utf-8"))
    print(f"{'stage':<12} {'trạng thái':<10} {'phút':>6}  ghi chú")
    print("-" * 78)
    for s in data["stages"]:
        print(f"{s['stage']:<12} {s['status']:<10} "
              f"{s.get('duration_sec', 0)/60:6.1f}  {s.get('note', '')[:44]}")
else:
    print("Chưa có smoke_run/reports/run_status.json — chạy cell smoke test trước.")

## 6 · Chạy thật

Chỉ chạy khi smoke test đã xanh. Sửa `ARGS` cho hợp với thời gian phiên:

| Phạm vi | `ARGS` | Ước lượng |
|---|---|---|
| Rất gọn | `--seeds 42 --backbones bert --variants resize --skip uos gas` | vài giờ |
| Vừa | `--seeds 42 --variants resize compact --skip uos` | ~1 phiên |
| Đầy đủ | `--skip uos` | nhiều phiên, cần `--resume` |

Ghi chú:

- `--resume` **luôn nên bật** để chạy tiếp ở phiên sau.
- `uos` cần Ollama chạy tại `localhost:11434` — Kaggle không có nên luôn bỏ qua;
  không bỏ thì stage tự skip kèm cảnh báo.
- Repo **không** chứa checkpoint (đã `.gitignore`) nên stage `ate` sẽ train T5 ATE
  từ đầu. Đó là điều kiện bắt buộc cho `ate_infer` và mọi eval end-to-end.

In [ ]:
ARGS = [
    "--seeds", "42",
    "--backbones", "bert",
    "--variants", "resize",
    "--skip", "uos",
    "--resume",
]

# Xem trước sẽ chạy những lệnh gì (không tốn GPU)
sh([sys.executable, "run_all.py", *ARGS, "--dry-run"], check=False)

In [ ]:
# ⚠️ Cell này chạy rất lâu. Kiểm tra lại ARGS ở cell trên trước khi chạy.
rc = sh([sys.executable, "run_all.py", *ARGS], check=False)
print(f"\nexit code: {rc}")

## 7 · Xem báo cáo

In [ ]:
from IPython.display import Markdown, display

# Đổi thành "smoke_run/reports" để xem báo cáo của lượt smoke
REPORT_DIR = REPO_DIR / "reports"

report = REPORT_DIR / "REPORT.md"
if report.is_file():
    display(Markdown(report.read_text(encoding="utf-8")))
else:
    print(f"Chưa có {report} — chạy `run_all.py --stages report` trước.")

In [ ]:
# Các bảng luận văn (nếu stage multiseed đã chạy)
tables = REPO_DIR / "runs_multiseed" / "thesis_tables.txt"
if tables.is_file():
    print(tables.read_text(encoding="utf-8"))
else:
    print(f"Chưa có {tables}")

## 8 · Đóng gói kết quả để tải về

Gom báo cáo + CSV + hình (KHÔNG gồm checkpoint `.pt`, quá nặng) thành một file zip
trong `/kaggle/working` để tải về từ tab Output.

In [ ]:
import zipfile
from datetime import datetime

PATTERNS = [
    "reports/**/*",
    "smoke_run/reports/**/*",
    "runs_ate/*.csv", "runs_ate/*.txt",
    "runs_joint/experiment_results_joint.*",
    "runs_joint_t5/experiment_results_joint.*",
    "runs_multiseed/*.csv", "runs_multiseed/*.txt",
    "runs_bert_gold/**/*.csv",
    "runs_gas/*.json", "runs_gas/*.csv",
    "thesis/figures/*.png", "thesis/figures/*.pdf",
]

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path = Path("/kaggle/working") / f"kltn_results_{stamp}.zip" \
    if Path("/kaggle/working").exists() else REPO_DIR / f"kltn_results_{stamp}.zip"

n = 0
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for pat in PATTERNS:
        for f in sorted(REPO_DIR.glob(pat)):
            if f.is_file() and f.suffix not in {".pt", ".bin", ".safetensors"}:
                zf.write(f, f.relative_to(REPO_DIR))
                n += 1

print(f"Đã đóng gói {n} file → {zip_path}")
print(f"Dung lượng: {zip_path.stat().st_size / 1024:.1f} KB")
print("\nTải về ở tab Output (panel bên phải) sau khi Save Version.")

---

# 9 · Chạy FULL trên dữ liệu đầy đủ

Các cell từ đây trở xuống dành cho lượt **thật**: dataset đầy đủ
(2 448 / 304 / 312 mẫu), epoch mặc định của từng script, 21 biến thể, 3 seed,
2 backbone. Không liên quan gì tới `--smoke`.

## Hai ràng buộc cứng của Kaggle

**1 · Thời gian — phiên tối đa 9–12 giờ.**
Lượt đầy đủ gồm 3 lượt train T5 ATE + 42 lượt train APC (stage `apc`) +
126 lượt train APC (stage `multiseed`). Không có cách nào nhét vừa một phiên.

**2 · Dung lượng — `/kaggle/working` chỉ 20 GB.**
Một checkpoint APC (`best_model.pt`) nặng **0,43 GB**, một checkpoint T5 nặng
**0,83 GB**:

| Stage | Số checkpoint | Dung lượng |
|---|---:|---:|
| `multiseed` | 126 | 54,5 GB |
| `apc` | 42 | 18,2 GB |
| `ate` | 3 | 2,5 GB |
| `gas` | 1 | 0,8 GB |
| **Tổng** | **172** | **75,9 GB** |

→ **gấp gần 4 lần hạn mức.** Giữ hết checkpoint là chắc chắn hết đĩa giữa chừng.

## Chiến lược: cắt lát + dọn checkpoint + nối phiên

Điều làm cho cách này chạy được: `common/run_multiseed.py` **eval ngay sau mỗi
lượt train** rồi ghi vào `runs_multiseed/results_raw.csv`, và lần chạy sau nó
bỏ qua combo đã có trong CSV đó (`done_keys`, kiểm tra *trước* khi train).
Nghĩa là **checkpoint của `multiseed` xoá được ngay sau mỗi lát** — chỉ cần
giữ `results_raw.csv` là resume được.

Checkpoint của stage `apc` thì phải giữ tới khi `gold` + `triplet` dùng xong,
nên ta chạy theo từng backbone rồi dọn.

| Phiên | Làm gì | Ước lượng |
|---|---|---|
| 1 | Giai đoạn 1 (ATE) + bắt đầu Giai đoạn 2 | ~9h |
| 2–3 | Giai đoạn 2 chạy tiếp (`results_raw.csv` lo phần nối) | ~9h/phiên |
| 4 | Giai đoạn 3 (apc + gold + triplet) | ~9h |
| 5 | Giai đoạn 4 (gas + results + figures + report) | ~4h |

## Cách nối hai phiên trên Kaggle

`/kaggle/working` **không** tự sống qua phiên. Quy trình:

1. Cuối phiên: bấm **Save Version → Save & Run All (Commit)**
2. Đợi commit xong, output của notebook thành một dataset
3. Phiên sau: **Add Input → Your Work → Notebook Output** (chọn version vừa commit)
4. Sửa `PREV_INPUT` ở cell ngay dưới trỏ vào `/kaggle/input/<slug>` rồi chạy


## 9.1 · Khôi phục trạng thái từ phiên trước

Bỏ qua cell này ở phiên đầu tiên (`PREV_INPUT = None`).


In [ ]:
# Trỏ vào output của phiên trước đã attach làm input.
# Xem tên chính xác bằng: !ls /kaggle/input
PREV_INPUT = None      # ví dụ: "/kaggle/input/kltn-token-merging-run-v3"

# Những thứ ĐỦ để resume — cố tình KHÔNG gồm best_model.pt của multiseed
RESTORE = [
    "runs_multiseed/results_raw.csv",      # ← quan trọng nhất: state resume của multiseed
    "runs_multiseed/results_aggregated.csv",
    "runs_ate",                            # prediction ATE theo seed + CSV tổng hợp
    "checkpoints",                         # checkpoint T5 ATE (cần cho ate_infer/results)
    "checkpoints_gas",
    "runs_joint", "runs_joint_t5",         # checkpoint APC (cần cho gold/triplet)
    "runs_bert_gold", "runs_gas", "reports",
]

if PREV_INPUT is None:
    print("PREV_INPUT = None → phiên đầu tiên, không khôi phục gì.")
    if Path("/kaggle/input").is_dir():
        found = sorted(p.name for p in Path("/kaggle/input").iterdir())
        print(f"Input đang attach: {found if found else '(trống)'}")
else:
    src_root = Path(PREV_INPUT)
    if not src_root.is_dir():
        raise SystemExit(f"Không thấy {src_root} — kiểm tra lại Add Input.")
    total = 0
    for rel in RESTORE:
        src = src_root / rel
        if not src.exists():
            continue
        dst = REPO_DIR / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True)
            n = sum(f.stat().st_size for f in src.rglob("*") if f.is_file())
        else:
            shutil.copy2(src, dst)
            n = src.stat().st_size
        total += n
        print(f"  khôi phục {rel:<38} {n / 2**30:6.2f} GB")
    print(f"\nTổng: {total / 2**30:.2f} GB")

    raw = REPO_DIR / "runs_multiseed" / "results_raw.csv"
    if raw.is_file():
        import csv as _csv
        rows = list(_csv.DictReader(raw.open(encoding="utf-8")))
        print(f"results_raw.csv: {len(rows)}/126 combo đã xong → sẽ được bỏ qua")


## 9.2 · Helper: giới hạn thời gian + dọn checkpoint

`run()` tự bỏ qua bước tiếp theo khi sắp hết giờ phiên, để notebook kịp chạy
cell lưu trạng thái ở cuối thay vì bị Kaggle giết ngang.


In [ ]:
import time

SESSION_HOURS = 8.5     # đặt thấp hơn hạn mức thật để còn thời gian commit
DEADLINE = time.time() + SESSION_HOURS * 3600

SEEDS     = ["42"]
BACKBONES = ["bert"]
GROUPS    = ["resize"]


def time_left_min():
    return (DEADLINE - time.time()) / 60


def disk_free_gb(path="/kaggle/working"):
    target = path if Path(path).exists() else str(REPO_DIR)
    return shutil.disk_usage(target).free / 2**30


def run(*args, need_min=25):
    """Gọi run_all.py, tự bỏ qua nếu không còn đủ thời gian."""
    left = time_left_min()
    if left < need_min:
        print(f"[HẾT GIỜ] còn {left:.0f} phút < {need_min} → bỏ qua: {' '.join(args)}")
        return None
    print(f"\n{'#' * 78}")
    print(f"# còn {left:.0f} phút | đĩa trống {disk_free_gb():.1f} GB | {' '.join(args)}")
    print(f"{'#' * 78}")
    return sh([sys.executable, "run_all.py", *args, "--resume"], check=False)


def prune(pattern, why=""):
    """Xoá checkpoint đã dùng xong để không hết đĩa."""
    n = freed = 0
    for f in sorted(REPO_DIR.glob(pattern)):
        if f.is_file():
            freed += f.stat().st_size
            f.unlink()
            n += 1
    print(f"[dọn] xoá {n} file ({freed / 2**30:.1f} GB) — {why}")
    print(f"[dọn] đĩa trống còn {disk_free_gb():.1f} GB")


print(f"Hạn phiên   : {SESSION_HOURS} giờ (còn {time_left_min():.0f} phút)")
print(f"Đĩa trống   : {disk_free_gb():.1f} GB")
print(f"Kế hoạch    : {len(SEEDS)} seed × {len(BACKBONES)} backbone × 21 biến thể")


## 9.3 · Giai đoạn 1 — ATE (T5 GAS)

Train T5 ATE cho cả 3 seed rồi sinh `test_ate_predictions.csv`. **Bắt buộc chạy
trước** mọi eval end-to-end. Repo không kèm checkpoint (đã `.gitignore`) nên
phiên đầu luôn phải train từ đầu.

Sinh ra `runs_ate/seed_<N>/test_predictions.csv` — nhờ đó stage `multiseed`
ghép cặp đúng ATE(seed N) ↔ APC(seed N) thay vì dùng chung một file.


In [ ]:
run("--stages", "env", "--seeds", *SEEDS)
run("--stages", "ate", "--seeds", *SEEDS, need_min=90)
run("--stages", "ate_infer", "--seeds", *SEEDS, need_min=15)

for s in SEEDS:
    f = REPO_DIR / "runs_ate" / f"seed_{s}" / "test_predictions.csv"
    print(f"  seed {s}: {'OK' if f.is_file() else 'THIẾU'}  {f}")


## 9.4 · Giai đoạn 2 — `multiseed` (126 lượt train, cắt lát)

Chạy từng lát `(seed × backbone × nhóm biến thể)` và **xoá checkpoint sau mỗi
lát**. Không xoá thì 54,5 GB checkpoint sẽ làm hết đĩa ở khoảng lát thứ 10.

An toàn vì `results_raw.csv` được ghi lại sau *mỗi* lượt train và là thứ duy
nhất cần để resume. Cell này chạy lại nhiều phiên đều được — lát nào xong rồi
sẽ tự `SKIP`.


In [ ]:
done, skipped = [], []

for seed in SEEDS:
    for bb in BACKBONES:
        for grp in GROUPS:
            tag = f"seed={seed} {bb} {grp}"
            rc = run("--stages", "multiseed",
                     "--seeds", seed, "--backbones", bb, "--variants", grp,
                     need_min=45)
            if rc is None:
                skipped.append(tag)
                continue
            done.append((tag, rc))
            # Checkpoint multiseed là dùng một lần — eval đã chạy inline xong.
            prune("runs_multiseed/**/best_model.pt", f"đã eval xong {tag}")

print(f"\n{'=' * 78}")
print(f"Đã chạy {len(done)} lát, bỏ qua {len(skipped)} lát vì hết giờ")
for tag, rc in done:
    print(f"  {'OK ' if rc == 0 else f'rc={rc}'}  {tag}")
if skipped:
    print("\nCòn lại cho phiên sau:")
    for tag in skipped:
        print(f"  - {tag}")


In [ ]:
# Tiến độ tổng: bao nhiêu trong 126 combo đã xong
import csv as _csv

raw = REPO_DIR / "runs_multiseed" / "results_raw.csv"
if raw.is_file():
    rows = list(_csv.DictReader(raw.open(encoding="utf-8")))
    keys = {(r["model_type"], r["config_id"], r["seed"]) for r in rows}
    print(f"{len(keys)}/126 combo đã xong ({len(keys) / 126 * 100:.0f}%)")
    from collections import Counter
    for bb, n in sorted(Counter(k[0] for k in keys).items()):
        print(f"  {bb}: {n}")
else:
    print("Chưa có runs_multiseed/results_raw.csv")


## 9.5 · Giai đoạn 3 — `apc` + `gold` + `triplet` theo từng backbone

Stage `apc` train 21 biến thể vào `runs_joint*/` và **giữ** checkpoint, vì
`gold` và `triplet` cần đọc. 21 × 0,43 = 9,1 GB mỗi backbone — vừa đĩa nếu làm
xong backbone nào thì dọn backbone đó.

Chạy xong Giai đoạn 2 hẵng chạy cell này. Nếu chỉ còn ít thời gian, đặt
`BACKBONES = ["bert"]` ở cell 9.2 rồi làm T5 ở phiên sau.


In [ ]:
for bb in BACKBONES:
    rc = run("--stages", "apc", "--backbones", bb, "--seeds", SEEDS[0], need_min=120)
    if rc is None:
        break
    run("--stages", "gold", "--backbones", bb, need_min=20)
    run("--stages", "triplet", "--backbones", bb, need_min=20)

    # gold + triplet đã đọc xong → checkpoint của backbone này không cần nữa.
    # Bỏ comment nếu sắp hết đĩa; giữ lại nếu còn muốn chạy `results`.
    # prune(f"{'runs_joint' if bb == 'bert' else 'runs_joint_t5'}/*/best_model.pt",
    #       f"xong gold+triplet cho {bb}")

print(f"\nĐĩa trống: {disk_free_gb():.1f} GB")


## 9.6 · Giai đoạn 4 — GAS, results, hình, báo cáo

- `gas` — train GAS một bước (sinh thẳng bộ ba) làm baseline đối chứng.
  Độc lập với ATE→APC, bỏ qua được.
- `results` — eval trên `results.csv`; cần checkpoint APC trong `runs_joint*/`
  nên phải chạy **trước** khi dọn ở Giai đoạn 3.
- `multiseed --no-train` — gộp lại **toàn bộ** bảng luận văn từ
  `results_raw.csv` đã tích luỹ. Bắt buộc chạy sau khi cắt lát, vì mỗi lát chỉ
  sinh bảng cho nhóm biến thể của riêng nó.


In [ ]:
run("--stages", "gas", "--seeds", SEEDS[0], need_min=60)
run("--stages", "results", need_min=30)
run("--stages", "figures", need_min=10)

# Gộp lại toàn bộ 8 bảng luận văn từ mọi lát đã chạy
run("--stages", "multiseed", "--no-train",
    "--seeds", *SEEDS, "--backbones", *BACKBONES, need_min=5)

run("--stages", "report", need_min=5)


## 9.7 · Lưu trạng thái cho phiên sau

Chạy cell này **trước khi** bấm Save Version. Nó gom đúng những thứ cần để
phiên sau resume, và cảnh báo nếu tổng dung lượng vượt hạn mức output.

Sau đó: **Save Version → Save & Run All (Commit)**, rồi ở phiên mới
**Add Input → Your Work → Notebook Output** và đặt `PREV_INPUT` ở cell 9.1.


In [ ]:
KEEP = [
    "runs_multiseed/results_raw.csv",
    "runs_multiseed/results_aggregated.csv",
    "runs_multiseed/thesis_tables.txt",
    "runs_multiseed/results_summary.txt",
    "runs_ate", "runs_bert_gold", "runs_gas", "reports",
    "checkpoints",       # T5 ATE — train lại rất tốn, nên giữ
    "checkpoints_gas",
    "runs_joint", "runs_joint_t5",
]

print(f"{'đường dẫn':<42} {'GB':>7}  ghi chú")
print("-" * 78)
total = 0
for rel in KEEP:
    p_ = REPO_DIR / rel
    if not p_.exists():
        print(f"{rel:<42} {'—':>7}  chưa có")
        continue
    size = (sum(f.stat().st_size for f in p_.rglob("*") if f.is_file())
            if p_.is_dir() else p_.stat().st_size)
    total += size
    print(f"{rel:<42} {size / 2**30:7.2f}")
print("-" * 78)
print(f"{'TỔNG':<42} {total / 2**30:7.2f} GB")

if total / 2**30 > 19:
    print("\n⚠️  Vượt ~20 GB hạn mức output của Kaggle.")
    print("   Dọn bớt checkpoint APC rồi chạy lại cell này:")
    print("   prune('runs_joint/*/best_model.pt', 'giai phong cho commit')")
    print("   prune('runs_joint_t5/*/best_model.pt', 'giai phong cho commit')")
else:
    print("\nVừa hạn mức. Bấm Save Version → Save & Run All (Commit).")

# Bản zip nhẹ chỉ gồm số liệu — luôn tải về được dù checkpoint có bị dọn
import zipfile
from datetime import datetime

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_zip = (Path("/kaggle/working") if Path("/kaggle/working").exists() else REPO_DIR)
out_zip = out_zip / f"kltn_state_{stamp}.zip"
n = 0
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for pat in ["runs_multiseed/*.csv", "runs_multiseed/*.txt",
                "runs_ate/**/*.csv", "runs_ate/*.txt",
                "runs_joint*/experiment_results_joint.*",
                "runs_joint*/*/meta.json",
                "runs_bert_gold/**/*.csv", "runs_gas/*",
                "reports/**/*", "thesis/figures/*"]:
        for f in sorted(REPO_DIR.glob(pat)):
            if f.is_file() and f.suffix not in {".pt", ".bin", ".safetensors"}:
                zf.write(f, f.relative_to(REPO_DIR))
                n += 1
print(f"\nZip số liệu: {n} file → {out_zip} ({out_zip.stat().st_size / 2**20:.1f} MB)")
